# Task 2: Build a Training Pipeline — Fine-tuning Whisper Large-v3 with QLoRA

This notebook implements the complete end-to-end training pipeline for fine-tuning **Whisper Large-v3** on the **Neyshekar Persian Speech ASR Dataset** using **4-bit QLoRA (BitsAndBytes)** and **Hugging Face Seq2SeqTrainer**.

### Pipeline Structure:
1. **`config.py`**: Centralized hyperparameter and seed management (`set_seed(42)`).
2. **`dataset.py`**: HuggingFace dataset loader and dynamic audio/text `DataCollatorSpeechSeq2SeqWithPadding`.
3. **`model.py`**: Base model loading in 4-bit NF4 quantization + LoRA adapter injection.
4. **`metrics.py`**: Normalized WER and CER calculation for Persian text.
5. **`train.py`**: Main execution script running `Seq2SeqTrainer`.

## Step 1: Install Dependencies (Google Colab Environment)

In [ ]:
# Install required libraries on Google Colab GPU environment
!pip install -q transformers datasets evaluate jiwer bitsandbytes peft accelerate soundfile num2fawords scikit-learn

## Step 2: Mount Google Drive & Navigate to Project Directory

In [ ]:
import os
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

# Navigate to project folder uploaded on Google Drive
PROJECT_PATH = '/content/drive/MyDrive/neyshekar_asr'
if os.path.exists(PROJECT_PATH):
    os.chdir(PROJECT_PATH)
    print(f"Successfully moved to project path: {PROJECT_PATH}")
else:
    print(f"[WARNING] Path '{PROJECT_PATH}' not found. Please ensure project folder is uploaded to MyDrive.")

## Step 3: Verify GPU Availability & Architecture

In [ ]:
import torch
print("PyTorch Version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Device Name:", torch.cuda.get_device_name(0))
    print("Total VRAM (GB):", round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2))

## Step 4: Execute Quick Smoke Test (30 Steps) to Verify Pipeline Stability

In [ ]:
# Run quick 30-step smoke test to confirm zero fine-tuning errors and verify WER/CER calculation
!python train.py --max_steps 30

## Step 5: Execute Full Fine-Tuning (3 Epochs)

In [ ]:
# Execute full 3-epoch training across 33,432 training samples
# !python train.py --epochs 3